In [41]:
from pathlib import Path
import pandas as pd


RAW_DATA_PATH = Path("../data/raw/customer_support_tickets.csv")
PROCESSED_DATA_PATH = Path("../data/processed/cleaned_tickets.csv")

In [42]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset shape: {df.shape}")

Raw dataset shape: (28587, 16)


In [43]:
required_columns = ["subject", "body", "queue", "language"]

print(f"Required columns available: {set(required_columns).issubset(df.columns)}")

Required columns available: True


In [44]:
#Filterning the dataset - We need only English tickets belonging to our four chosen queues

SELECTED_QUEUES = [
    "Billing and Payments",
    "Customer Service",
    "Product Support",
    "Technical Support",
]

filtered_df = df[(df["language"] == "en") & (df["queue"].isin(SELECTED_QUEUES))].copy()

print(f"Filtered dataset shape: {filtered_df.shape}")
print(f"Languages: {filtered_df['language'].unique().tolist()}")
print(f"Queues: {sorted(filtered_df['queue'].unique().tolist())}")#

Filtered dataset shape: (11815, 16)
Languages: ['en']
Queues: ['Billing and Payments', 'Customer Service', 'Product Support', 'Technical Support']


In [45]:
#Adding assertions - An assert stops execution if an expected condition is false.
#No output means both validations passed.

assert filtered_df["language"].eq("en").all()
assert set(filtered_df["queue"].unique()) == set(SELECTED_QUEUES)

In [46]:
#Inspect missing values and blank text

filtered_df[["subject", "body", "queue"]].isna().sum()

subject    1842
body          0
queue         0
dtype: int64

In [47]:
#Check blank or whitespace-only values

text_columns = ["subject", "body"]

blank_counts = (
    filtered_df[text_columns]
    .fillna("")
    .apply(lambda column: column.str.strip().eq("").sum())
)

blank_counts

subject    1842
body          0
dtype: int64

In [48]:
cleaned_df = filtered_df.copy()

In [49]:
#Replacing missing subjects with an empty string

cleaned_df["subject"] = cleaned_df["subject"].fillna("")

In [50]:
#Removing unnecessary whitespace

cleaned_df["subject"] = cleaned_df["subject"].str.strip()
cleaned_df["body"] = cleaned_df["body"].str.strip()

In [51]:
#Combining two columns

cleaned_df["ticket_text"] = (cleaned_df["subject"] + " " + cleaned_df["body"]).str.strip()

#identifying malformed tickets - ref from 03_exploratory_data_analysis notebook

word_count = cleaned_df["ticket_text"].str.split().str.len()
character_count = cleaned_df["ticket_text"].str.len()

malformed_mask = (
    (word_count < 5)
    & (character_count > 100)
)

print(f"Clearly malformed tickets: {malformed_mask.sum()}")



Clearly malformed tickets: 1


In [52]:
cleaned_df[["subject", "body", "ticket_text"]].head()

,subject,body,ticket_text
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,..."
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope..."
7,Connectivity Problems with Printer on MacBook Pro,"Dear Support Team,\n\nI am reporting a recurri...",Connectivity Problems with Printer on MacBook ...
10,VPN Access Issue,"Customer Support,\n\nWe are encountering a dis...","VPN Access Issue Customer Support,\n\nWe are e..."


In [53]:
#Count empty combined inputs
cleaned_df["ticket_text"].eq("").sum()

np.int64(0)

In [54]:
#Check records whose original subject was missing:

cleaned_df.loc[filtered_df["subject"].isna(),["subject", "body", "ticket_text"]].head()



,subject,body,ticket_text
870,,"Customer Support, I am requesting comprehensiv...","Customer Support, I am requesting comprehensiv..."
887,,"Customer Support,\n\nI am reporting a critical...","Customer Support,\n\nI am reporting a critical..."
915,,"Customer Support, drafting a request for enhan...","Customer Support, drafting a request for enhan..."
929,,Customer support has received a report regardi...,Customer support has received a report regardi...
946,,"Dear Customer Support, I am reaching out to re...","Dear Customer Support, I am reaching out to re..."


In [55]:
#Detect duplicate tickets

duplicate_count = cleaned_df.duplicated(subset=["ticket_text"]).sum()

print(f"Duplicate ticket texts: {duplicate_count}")

Duplicate ticket texts: 0


In [56]:
duplicate_rows = cleaned_df[
    cleaned_df.duplicated(subset=["ticket_text"], keep=False)
].sort_values("ticket_text")

duplicate_rows[["ticket_text", "queue"]].head(10)

,ticket_text,queue


In [57]:
#Creating the modeling DataFrame and reset index
#The ~ means “not,” so this keeps rows that do not match the malformed-ticket condition.

model_df = (cleaned_df.loc[~malformed_mask,["ticket_text", "queue"],]
    .reset_index(drop=True))
model_df.head()

,ticket_text,queue
0,"Account Disruption Dear Customer Support Team,...",Technical Support
1,Inquiry Regarding Invoice Details Dear Custome...,Billing and Payments
2,"Feature Query Dear Customer Support,\n\nI hope...",Technical Support
3,Connectivity Problems with Printer on MacBook ...,Technical Support
4,"VPN Access Issue Customer Support,\n\nWe are e...",Product Support


In [58]:
model_df.shape

(11814, 2)

In [59]:
#Save the processed dataset

print(f"Malformed rows: {malformed_mask.sum()}")
print(f"Cleaned rows before removal: {cleaned_df.shape[0]}")
print(f"Model rows after removal: {model_df.shape[0]}")
print(f"Saving to: {PROCESSED_DATA_PATH.resolve()}")


model_df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Processed file exists: {PROCESSED_DATA_PATH.exists()}")
print(
    f"Processed file size: "
    f"{PROCESSED_DATA_PATH.stat().st_size / (1024 * 1024):.2f} MB"
)

Malformed rows: 1
Cleaned rows before removal: 11815
Model rows after removal: 11814
Saving to: /Users/yogithagujjarlapudi/AI-ML Course/AI-ML-PROJECTS/AI-ML/Customer-Support-Ticket-Intelligence/data/processed/cleaned_tickets.csv
Processed file exists: True
Processed file size: 4.81 MB


In [60]:
#Reload the df

verification_df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"Reloaded shape: {verification_df.shape}")
print(f"Reloaded columns: {verification_df.columns.tolist()}")
print(f"Missing values: {verification_df.isna().sum().sum()}")
print(f"Empty ticket texts: {verification_df['ticket_text'].eq('').sum()}")

Reloaded shape: (11814, 2)
Reloaded columns: ['ticket_text', 'queue']
Missing values: 0
Empty ticket texts: 0


In [61]:
assert verification_df.equals(model_df)